# Previsão de viagens

## Preparar os dados
Em uma pasta nova, execute a célula abaixo. Depois procure o título de sua aula. Tente resolver cada atividade antes de executar sua solução.

In [ ]:
from pathlib import Path
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["scikit-learn", "matplotlib"])
arquivos = {'laboratorio.py': '"""Dados cronológicos e ferramentas compartilhadas pelas aulas. Execute na raiz do kit."""\nfrom pathlib import Path\nimport csv\nimport numpy as np\nfrom sklearn.dummy import DummyRegressor\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.pipeline import make_pipeline\n\nDADOS = Path(__file__).resolve().parent / "dados"\n\n\ndef carregar(nome="viagens.csv"):\n    with (DADOS / nome).open(encoding="utf-8", newline="") as arquivo:\n        linhas = sorted(csv.DictReader(arquivo), key=lambda item: (item["data"], item["id"]))\n    ids = [linha["id"] for linha in linhas]\n    if len(ids) != len(set(ids)):\n        raise ValueError("ID repetido")\n    X = np.array([[float(r["km_planejados"] or "nan"), float(r["paradas_planejadas"])] for r in linhas])\n    y = np.array([float(r["minutos_reais"]) for r in linhas])\n    return ids, X, y\n\n\ndef alternativas():\n    # A mediana minimiza a soma dos erros absolutos de uma previsão constante no treino.\n    return {"mediana": DummyRegressor(strategy="median"),\n            "ridge-1": make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=1)),\n            "ridge-10": make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=10))}\n', 'dados/viagens.csv': 'id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\n', 'dados/serie.csv': 'id,data,km_planejados,paradas_planejadas,minutos_reais\nS001,2026-02-01,,0,15.1\nS002,2026-02-02,16,2,30.7\nS003,2026-02-03,4,3,14.3\nS004,2026-02-04,16,0,28.8\nS005,2026-02-05,2,2,7.6\nS006,2026-02-06,15,2,32.6\nS007,2026-02-07,15,3,28.5\nS008,2026-02-08,3,2,12.9\nS009,2026-02-09,8,6,32.6\nS010,2026-02-10,16,5,32.6\nS011,2026-02-11,13,2,29.0\nS012,2026-02-12,,6,27.7\nS013,2026-02-13,10,2,21.2\nS014,2026-02-14,9,4,19.1\nS015,2026-02-15,7,4,22.0\nS016,2026-02-16,4,5,21.7\nS017,2026-02-17,8,1,15.4\nS018,2026-02-18,17,6,39.0\nS019,2026-02-19,9,2,23.0\nS020,2026-02-20,15,0,23.9\nS021,2026-02-21,15,3,33.9\nS022,2026-02-22,4,2,11.5\nS023,2026-02-23,,1,25.6\nS024,2026-02-24,6,1,18.7\nS025,2026-02-25,5,6,31.3\nS026,2026-02-26,14,3,28.6\nS027,2026-02-27,10,5,26.1\nS028,2026-02-28,5,0,14.4\nS029,2026-03-01,10,4,17.7\nS030,2026-03-02,8,2,22.4\nS031,2026-03-03,3,1,15.6\nS032,2026-03-04,8,6,31.4\nS033,2026-03-05,6,3,24.5\nS034,2026-03-06,,4,33.0\nS035,2026-03-07,6,0,15.8\nS036,2026-03-08,6,4,21.7\nS037,2026-03-09,9,6,23.3\nS038,2026-03-10,14,1,16.1\nS039,2026-03-11,14,6,36.2\nS040,2026-03-12,15,2,28.1\nS041,2026-03-13,14,2,29.9\nS042,2026-03-14,13,1,29.7\nS043,2026-03-15,16,3,31.0\nS044,2026-03-16,17,2,32.4\nS045,2026-03-17,,4,30.2\nS046,2026-03-18,13,5,32.1\nS047,2026-03-19,4,0,8.3\nS048,2026-03-20,13,4,28.9\nS049,2026-03-21,4,5,27.0\nS050,2026-03-22,14,5,33.2\nS051,2026-03-23,16,3,32.7\nS052,2026-03-24,8,5,26.5\nS053,2026-03-25,5,0,8.7\nS054,2026-03-26,17,4,40.1\nS055,2026-03-27,12,0,24.8\nS056,2026-03-28,,5,38.1\nS057,2026-03-29,13,0,24.7\nS058,2026-03-30,18,5,37.2\nS059,2026-03-31,5,1,12.9\nS060,2026-04-01,16,4,29.6\nS061,2026-04-02,14,1,26.1\nS062,2026-04-03,2,6,16.6\nS063,2026-04-04,13,4,33.2\nS064,2026-04-05,5,5,24.6\nS065,2026-04-06,17,0,25.2\nS066,2026-04-07,17,6,40.2\nS067,2026-04-08,,6,27.2\nS068,2026-04-09,12,1,19.5\nS069,2026-04-10,10,6,31.6\nS070,2026-04-11,3,4,20.2\nS071,2026-04-12,14,1,24.1\nS072,2026-04-13,8,5,24.2\nS073,2026-04-14,6,3,17.6\nS074,2026-04-15,2,0,9.5\nS075,2026-04-16,9,1,20.6\nS076,2026-04-17,12,1,28.0\nS077,2026-04-18,7,0,14.2\nS078,2026-04-19,,1,20.2\nS079,2026-04-20,18,5,35.1\nS080,2026-04-21,6,5,30.2\nS081,2026-04-22,14,2,27.9\nS082,2026-04-23,7,1,23.9\nS083,2026-04-24,11,4,31.3\nS084,2026-04-25,15,3,31.7\nS085,2026-04-26,7,0,16.7\nS086,2026-04-27,2,6,22.1\nS087,2026-04-28,5,0,11.4\nS088,2026-04-29,2,6,22.7\nS089,2026-04-30,,4,25.5\nS090,2026-05-01,16,2,33.2\nS091,2026-05-02,12,4,28.3\nS092,2026-05-03,12,5,29.4\nS093,2026-05-04,2,4,16.0\nS094,2026-05-05,11,6,33.7\nS095,2026-05-06,6,5,25.7\nS096,2026-05-07,15,0,28.9\nS097,2026-05-08,10,5,28.4\nS098,2026-05-09,10,6,33.9\nS099,2026-05-10,3,1,14.7\nS100,2026-05-11,,1,32.1\nS101,2026-05-12,15,3,34.8\nS102,2026-05-13,3,2,15.3\nS103,2026-05-14,10,4,30.0\nS104,2026-05-15,14,2,27.6\nS105,2026-05-16,18,6,51.7\nS106,2026-05-17,18,6,41.6\nS107,2026-05-18,11,4,28.9\nS108,2026-05-19,6,0,18.8\nS109,2026-05-20,5,5,20.4\nS110,2026-05-21,11,1,19.5\nS111,2026-05-22,,5,42.0\nS112,2026-05-23,9,1,24.6\nS113,2026-05-24,11,0,26.2\nS114,2026-05-25,12,4,32.4\nS115,2026-05-26,11,1,20.9\nS116,2026-05-27,12,3,30.8\nS117,2026-05-28,9,3,28.8\nS118,2026-05-29,3,4,19.3\nS119,2026-05-30,15,4,35.1\nS120,2026-05-31,11,1,22.6\n', 'inicial/projeto.py': '"""Execute python -m inicial.projeto. Substitua cada TODO e explique suas escolhas."""\nfrom laboratorio import alternativas, carregar\nfrom sklearn.metrics import mean_absolute_error\n\nids, X, y = carregar("serie.csv")\nprint("Desenvolvimento:", ids[0], ids[99], "• Avaliação reservada:", ids[100], ids[-1])\nbase = alternativas()["mediana"].fit(X[:40], y[:40])\nprint("Primeira janela, MAE de referência (min):", mean_absolute_error(y[40:60], base.predict(X[40:60])))\n# TODO: repita as três janelas (cortes 40, 60, 80, horizonte 20) para cada alternativa.\n# TODO: congele o menor MAE médio, reajuste nos 100 dias e avalie nas 20 linhas finais.\n# TODO: desenhe os resíduos e indique uma limitação dos dados sintéticos.\n', 'solucao/projeto.py': '"""Execute python -m solucao.projeto a partir da raiz do kit."""\nfrom pathlib import Path\nimport numpy as np\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom sklearn.metrics import mean_absolute_error\nfrom laboratorio import alternativas, carregar\n\n\ndef selecionar(X, y):\n    # A função só recebe as 100 datas de desenvolvimento.\n    erros = {}\n    for nome, modelo in alternativas().items():\n        janelas = []\n        for corte in (40, 60, 80):\n            modelo.fit(X[:corte], y[:corte])\n            previsao = modelo.predict(X[corte:corte + 20])\n            janelas.append(mean_absolute_error(y[corte:corte + 20], previsao))\n        erros[nome] = float(np.mean(janelas))\n    nome = min(erros, key=erros.get)\n    return nome, alternativas()[nome].fit(X, y), erros\n\n\ndef avaliar(saida=Path("residuos.png")):\n    ids, X, y = carregar("serie.csv")\n    nome, vencedor, erros = selecionar(X[:100], y[:100])\n    previstos = vencedor.predict(X[100:])\n    ref = alternativas()["mediana"].fit(X[:100], y[:100]).predict(X[100:])\n    print("MAE de validação (min):", {k: round(v, 2) for k, v in erros.items()})\n    print(f"Congelado: {nome}; período final: {ids[100]} a {ids[-1]}")\n    for rotulo, resultado in [("selecionado", previstos), ("referência", ref)]:\n        print(f"MAE final {rotulo}: {mean_absolute_error(y[100:], resultado):.2f} min")\n    fig, ax = plt.subplots()\n    ax.scatter(np.arange(101, len(ids) + 1), y[100:] - previstos)\n    ax.axhline(0, color="black")\n    ax.set(xlabel="Viagem no período final", ylabel="Real − previsto (min)")\n    fig.savefig(saida, bbox_inches="tight")\n    plt.close(fig)\n    return previstos\n\n\nif __name__ == "__main__":\n    avaliar()\n', 'test_projeto.py': 'from pathlib import Path\nfrom tempfile import TemporaryDirectory\nimport unittest\nimport numpy as np\nfrom sklearn.metrics import mean_absolute_error\nfrom laboratorio import alternativas, carregar\nfrom solucao.projeto import avaliar, selecionar\n\n\nclass Viagens(unittest.TestCase):\n    def test_ponte_python(self):\n        ids, X, y = carregar()\n        self.assertEqual((len(ids), len(set(ids))), (10, 10))\n        previsoes = alternativas()["mediana"].fit(X[:6], y[:6]).predict(X[6:8])\n        self.assertEqual(mean_absolute_error(y[6:8], previsoes), 3.5)\n\n    def test_reserva(self):\n        _, X, y = carregar("serie.csv")\n        nome, modelo, erros = selecionar(X[:100], y[:100])\n        previsto = modelo.predict(X[100:])\n        y[100:] += 7\n        outro, novo, medidas = selecionar(X[:100], y[:100])\n        self.assertEqual((nome, erros), (outro, medidas))\n        np.testing.assert_allclose(previsto, novo.predict(X[100:]))\n        with TemporaryDirectory() as pasta:\n            destino = Path(pasta) / "residuos.png"\n            np.testing.assert_allclose(avaliar(destino), previsto)\n            self.assertTrue(destino.read_bytes().startswith(b"\\x89PNG"))\n'}
for nome, texto in arquivos.items():
    destino = Path(nome)
    if destino.exists() and destino.read_text(encoding="utf-8") != texto:
        raise FileExistsError(f"Use uma pasta nova para preservar {nome}")
    destino.parent.mkdir(exist_ok=True, parents=True)
    destino.write_text(texto, encoding="utf-8")
print("Laboratório preparado.")


# Prever antes da partida

Uma pessoa solicita a duração de uma viagem ao embarcar. A unidade de observação é uma viagem e a saída tem unidade de minutos. Um campo perfeitamente correlacionado com a duração seria inútil em produção se só fosse preenchido após a chegada.

### Demonstração

Inspecione o primeiro registro. Quais duas entradas poderiam estar disponíveis nesse instante?

In [ ]:
import csv
from laboratorio import DADOS
with (DADOS / "viagens.csv").open(encoding="utf-8") as f:
    viagem = next(csv.DictReader(f))
print(viagem["km_planejados"], viagem["paradas_planejadas"])

Saem 2 e 0. É possível conhecer o itinerário planejado antes de percorrê-lo.

### Variação

Mostre o valor faturado e o alvo real. A igualdade permite usar o faturamento para prever?

In [ ]:
print(viagem["minutos_faturados"], viagem["minutos_reais"])

Ambos valem 6 na primeira viagem. Faturamento só existe depois da chegada e introduziria vazamento.

### Aplicação

Descreva uma previsão de S101 sem ler o alvo. Identifique as colunas copiadas para X.

In [ ]:
from laboratorio import carregar
ids, X, y = carregar("serie.csv")
print(ids[100], X[100].tolist())

A linha contém distância e paradas planejadas. O ID rastreia a viagem. A previsão pretendida é a duração antes da partida.

# Separar observações no tempo

Um teste cronológico imita o uso do modelo no futuro. Ordene uma única vez e corte IDs, atributos e alvos nos mesmos índices. Nas dez viagens válidas da trilha Python, os seis primeiros dias ensinam o modelo, os dois seguintes apoiam escolhas e os dois últimos ficam reservados.

### Demonstração

Localize os IDs de cada partição antes de comparar os tamanhos.

In [ ]:
from laboratorio import carregar
ids, X, y = carregar()
print(ids[:6], ids[6:8], ids[8:])

Treino termina em V06. Validação contém V07 e V08. O teste contém V09 e V10.

### Variação

Mostre por que iniciar a validação em 5 contaminaria a comparação.

In [ ]:
print(set(ids[:6]) & set(ids[5:8]))

V06 aparece nos dois lados e deixa de simular uma viagem inédita.

### Aplicação

Verifique a disjunção dos três grupos e a cobertura das dez viagens.

In [ ]:
a, b, c = map(set, (ids[:6], ids[6:8], ids[8:]))
print(a.isdisjoint(b), b.isdisjoint(c), a.isdisjoint(c), len(a | b | c))

O resultado tem três True e 10 IDs distintos. Usar os mesmos cortes em X e y mantém cada alvo ligado à sua viagem.

# Construir a previsão de referência

Antes de treinar um modelo com atributos, use uma regra simples que só aprende um número. A mediana dos alvos de treino é uma referência apropriada para minimizar a soma dos desvios absolutos no treino. Conhecer as respostas da validação para escolher a constante favoreceria a referência indevidamente.

### Demonstração

Ache o valor central dos seis tempos no treino.

In [ ]:
import numpy as np
from laboratorio import carregar, alternativas
ids, X, y = carregar()
print(y[:6].tolist(), np.median(y[:6]))

Os tempos são 6, 8, 10, 10, 12, 14. Os dois centrais valem 10.

### Variação

Ajuste a referência e preveja para duas viagens com entradas diferentes.

In [ ]:
base = alternativas()["mediana"].fit(X[:6], y[:6])
print(base.predict(X[6:8]).tolist())

A previsão é [10.0, 10.0]. DummyRegressor com estratégia median ignora as diferenças entre os atributos.

### Aplicação

Mostre o efeito de trocar um único alvo do treino, de 14 para 140. Compare média e mediana.

In [ ]:
alterado = y[:6].copy()
alterado[-1] = 140
print(np.mean(alterado), np.median(alterado))

A média passa para 31, enquanto a mediana permanece 10. Um valor extremo pode mover fortemente a média.

# Interpretar o erro em minutos

O erro absoluto é a distância entre uma previsão e o valor medido. O MAE é a média dessas distâncias. Dois desvios em sentidos opostos contam para o erro sem se anularem. A unidade é a mesma do alvo.

### Demonstração

Reconstrua o MAE de duas previsões constantes sem biblioteca de métricas.

In [ ]:
import numpy as np
from laboratorio import carregar
_, X, y = carregar()
erros = np.abs(y[6:8] - 10)
print(erros.tolist(), erros.mean())

A referência erra por 2 e 5 minutos, com média de 3,5.

### Variação

Compare com a média dos desvios que preservam sinal.

In [ ]:
print((y[6:8] - 10).mean())

O resultado é 1,5. A subestimação de uma viagem compensa em parte a superestimação da outra.

### Aplicação

Um candidato prevê 9 e 13 nos mesmos dias. Calcule seu MAE e decida qual vence nesta janela.

In [ ]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y[6:8], [9, 13]))

O candidato obtém 1,5 min e vence a referência de 3,5 nesta validação. Duas viagens ainda dão pouca informação sobre o futuro.

# Alinhar atributos e alvo

Para aprender uma relação numérica, o estimador recebe uma matriz X com uma linha por viagem e um vetor y na mesma ordem. Comece pela contagem de paradas, que está preenchida nas dez linhas. Uma matriz com uma única coluna continua bidimensional.

### Demonstração

Confira dimensões e rastreie a terceira observação.

In [ ]:
from laboratorio import carregar
ids, X, y = carregar()
paradas = X[:, 1:2]
print(paradas.shape, y.shape, ids[2], paradas[2, 0], y[2])

Há 10 linhas e uma coluna de paradas. V03 tem uma parada e alvo de 10 minutos.

### Variação

Inspecione a forma ao selecionar um índice de coluna em vez de uma fatia.

In [ ]:
print(X[:, 1].shape, X[:, 1:2].shape)

O índice produz (10,) e a fatia (10, 1). O ajuste de regressão espera a segunda forma.

### Aplicação

Extraia os atributos e alvos da janela de validação e mostre o vínculo por ID.

In [ ]:
print(list(zip(ids[6:8], paradas[6:8, 0].tolist(), y[6:8].tolist())))

V07 tem uma parada e dura 8 minutos. V08 tem zero e dura 15. Cortar cada vetor na mesma posição conserva o vínculo.

# Ajustar uma reta

Uma regressão linear com um atributo aprende uma inclinação e um intercepto. O ajuste por mínimos quadrados escolhe parâmetros que minimizam a soma dos quadrados dos resíduos no treino. A métrica usada depois para avaliar pode ser MAE.

### Demonstração

Ajuste usando só o treino e imprima os dois parâmetros.

In [ ]:
from sklearn.linear_model import LinearRegression
from laboratorio import carregar
_, X, y = carregar()
modelo = LinearRegression().fit(X[:6, 1:2], y[:6])
print(round(modelo.intercept_, 2), round(modelo.coef_[0], 2))

O intercepto estima a duração com zero paradas. A inclinação tem unidade de minutos por parada adicional.

### Variação

Preveja uma viagem com duas paradas pela equação.

In [ ]:
print(round(modelo.intercept_ + 2 * modelo.coef_[0], 2))

A equação multiplica a inclinação pelo número de paradas e soma o intercepto.

### Aplicação

Verifique o significado da inclinação calculando a diferença entre previsões com duas e três paradas.

In [ ]:
import numpy as np
p = modelo.predict([[2], [3]])
print(np.isclose(p[1] - p[0], modelo.coef_[0]))

A verificação imprime True. Uma associação no treino, isoladamente, não demonstra que acrescentar uma parada causará esse atraso.

# Prever sem reajustar

predict aplica os parâmetros aprendidos a novas linhas com o mesmo contrato de entrada. Para comparar candidatos, ambos devem receber exatamente as viagens de validação e ser avaliados contra seus respectivos alvos.

### Demonstração

Gere previsões para V07 e V08 sem passar os tempos reais a predict.

In [ ]:
from laboratorio import carregar
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
ids, X, y = carregar()
modelo = LinearRegression().fit(X[:6, 1:2], y[:6])
p = modelo.predict(X[6:8, 1:2])
print(list(zip(ids[6:8], p.round(2).tolist())))

Saem duas estimativas na ordem V07, V08. Observe que a viagem com zero paradas recebe a estimativa do intercepto.

### Variação

Calcule a distância média da previsão ao alvo.

In [ ]:
print(round(mean_absolute_error(y[6:8], p), 2))

Compare o valor com 3,5 minutos. Aqui a reta com paradas tem MAE maior que a referência.

### Aplicação

Refaça a comparação com as linhas da validação em ordem inversa. Preserve os pares.

In [ ]:
print(round(mean_absolute_error(y[6:8][::-1], p[::-1]), 2))

O MAE é idêntico porque cada previsão ainda acompanha o mesmo alvo.

# Investigar o maior resíduo

Defina resíduo como valor observado menos previsão. Sinal positivo indica que a viagem demorou mais que o previsto. Procure a maior magnitude e recupere o ID para investigar. Um valor extremo em janela pequena merece contexto.

### Demonstração

Rastreie o maior erro da regressão de paradas.

In [ ]:
import numpy as np
from laboratorio import carregar
from sklearn.linear_model import LinearRegression
ids, X, y = carregar()
m = LinearRegression().fit(X[:6, 1:2], y[:6])
r = y[6:8] - m.predict(X[6:8, 1:2])
i = np.argmax(np.abs(r))
print(ids[6:8][i], round(r[i], 2))

V08 tem duração subestimada. O resíduo é positivo mesmo sem paradas planejadas.

### Variação

O que acontece com o MAE se alguém trocar a ordem da subtração?

In [ ]:
print(np.abs(r).mean() == np.abs(-r).mean())

Imprime True. Os sinais se invertem, mas as magnitudes são as mesmas.

### Aplicação

Produza um diagnóstico textual para cada viagem e indique uma investigação posterior.

In [ ]:
for chave, valor in zip(ids[6:8], r):
    print(chave, "subestimada" if valor > 0 else "superestimada" if valor < 0 else "exata")

As duas linhas mostram sentidos distintos. Investigue se a duração depende também do percurso e quais atributos estariam disponíveis antecipadamente.

# Aprender a preencher ausências

Uma distância ausente representa uma medida desconhecida. Zero é um valor numérico diferente. Aprenda a mediana da distância apenas com o treino e guarde a mesma regra para transformar os dias seguintes.

### Demonstração

Ajuste o imputador e localize a distância substituída na terceira viagem.

In [ ]:
from sklearn.impute import SimpleImputer
from laboratorio import carregar
_, X, y = carregar()
im = SimpleImputer(strategy="median").fit(X[:6])
print(im.statistics_.tolist(), im.transform(X[2:3]).tolist())

A mediana de distância é 4 km. A ausência em V03 passa a 4 e sua contagem de paradas permanece 1.

### Variação

Transforme as viagens de validação. V07, com 100 km, deve mudar a mediana?

In [ ]:
print(im.transform(X[6:8]).tolist(), im.statistics_[0])

V07 mantém 100. A distância de V08 passa a 4. Transformar o lote conserva a estatística aprendida.

### Aplicação

Altere só as distâncias de um lote futuro e confirme a independência do ajuste anterior.

In [ ]:
futuro = X[8:].copy()
futuro[:, 0] = 999
im.transform(futuro)
print(im.statistics_[0])

Permanece 4. Um novo fit no lote futuro misturaria informações de momentos diferentes.

# Compor transformação e modelo

Pipeline encadeia etapas. Durante fit, cada transformação é aprendida no treino. Durante predict, o mesmo imputador transforma a entrada e a regressão produz uma previsão. A composição evita esquecer uma etapa na avaliação.

### Demonstração

Monte e ajuste um pipeline que aceite distâncias ausentes.

In [ ]:
from laboratorio import carregar
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
_, X, y = carregar()
modelo = make_pipeline(SimpleImputer(strategy="median"), Ridge(alpha=1))
modelo.fit(X[:6], y[:6])
print(modelo.predict(X[7:8]).round(2).tolist())

Há uma previsão finita para V08 apesar do valor ausente na entrada.

### Variação

Inspecione as etapas e a estatística da primeira.

In [ ]:
print(list(modelo.named_steps), modelo[0].statistics_.tolist())

A primeira é simpleimputer. Sua mediana da primeira coluna é 4. A última é ridge.

### Aplicação

Faça inferência em uma viagem inédita de 4 km com duas paradas e confira o formato.

In [ ]:
print(modelo.predict([[4, 2]]).shape)

Uma linha de entrada produz um vetor com uma previsão. A ordem das colunas continua distância, paradas.

# Validar no futuro de cada janela

Uma única janela pode ser atípica. Na série maior, valide em três blocos consecutivos de 20 dias. Em cada rodada, todos os dias do treino são anteriores ao bloco de validação. Os primeiros treinos reaparecem expandidos. Os 20 dias finais ficam de fora desta escolha.

### Demonstração

Construa os limites e mostre os IDs de cada avaliação.

In [ ]:
from laboratorio import carregar
ids, X, y = carregar("serie.csv")
for corte in (40, 60, 80):
    print(ids[corte - 1], ids[corte], ids[corte + 19])

A primeira linha é S040, S041, S060. A última termina em S100.

### Variação

Reproduza as mesmas divisões com TimeSeriesSplit.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
for tr, va in TimeSeriesSplit(n_splits=3, test_size=20).split(X[:100]):
    print(len(tr), len(va), tr[-1] < va[0])

São treinos com 40, 60 e 80 linhas. Cada validação contém 20 e ocorre depois de seu treino.

### Aplicação

Calcule três MAEs da referência. Que número usaria para compará-la com outro candidato?

In [ ]:
from laboratorio import alternativas
from sklearn.metrics import mean_absolute_error
for c in (40, 60, 80):
    ref = alternativas()["mediana"].fit(X[:c], y[:c])
    print(round(mean_absolute_error(y[c:c+20], ref.predict(X[c:c+20])), 2))

Use a média dos três MAEs. Como os blocos têm o mesmo tamanho, cada viagem recebe o mesmo peso.

# Escolher um hiperparâmetro

Ridge acrescenta uma penalidade aos coeficientes. alpha controla a intensidade da penalidade. Decida quais alternativas comparar antes de olhar o período final. Os coeficientes são aprendidos pelo ajuste e o valor de alpha é escolhido pela validação.

### Demonstração

Compare os dois alphas e a referência nas janelas já definidas. Inspecione a implementação de selecionar no kit.

In [ ]:
from laboratorio import carregar
from solucao.projeto import selecionar
_, X, y = carregar("serie.csv")
nome, m, erros = selecionar(X[:100], y[:100])
print(nome, {k: round(v, 2) for k, v in erros.items()})

ridge-1 obtém cerca de 2,60 minutos de MAE médio, pouco abaixo de ridge-10. A solução reajusta o escolhido nos cem dias.

### Variação

Altere somente um alvo reservado e compare o dicionário de erros.

In [ ]:
alternativo = y.copy()
alternativo[-1] += 500
print(selecionar(X[:100], alternativo[:100])[2] == erros)

True indica que a avaliação reservada não entrou nas escolhas.

### Aplicação

Verifique se a mediana do imputador do modelo escolhido coincide com a mediana dos km do desenvolvimento.

In [ ]:
import numpy as np
print(np.isclose(m[0].statistics_[0], np.nanmedian(X[:100, 0])))

True. No reajuste final, o imputador aproveita todas as cem linhas disponíveis antes da reserva.

# Definir a classe positiva

Uma equipe quer decidir se deve inspecionar uma bateria com base na vibração medida agora. Cada linha abaixo representa uma bateria diferente. Classe 1 indica que a inspeção posterior confirmou desgaste. Aqui o alvo é uma categoria binária. Os registros estão em ordem de coleta.

### Demonstração

Separe as oito medições de treino e as quatro de validação sem trocar os rótulos.

In [ ]:
import numpy as np
sinal = np.array([1, 2, 3, 4, 6, 7, 8, 9, 2, 4, 7, 9]).reshape(-1, 1)
desgaste = np.array([0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1])
print(sinal[:8].shape, desgaste[8:].tolist())

Em validação, a inspeção confirmou desgaste na segunda e na quarta baterias.

### Variação

Conte positivos em cada partição antes de treinar.

In [ ]:
print(desgaste[:8].sum(), desgaste[8:].sum())

Há quatro positivos no treino e dois na validação. O treino contém exemplos de ambas as classes.

### Aplicação

Uma regra sempre diz “sem desgaste”. Quantas inspeções necessárias ela perde na validação?

In [ ]:
print(np.count_nonzero(desgaste[8:] == 1))

Perde duas. Mesmo com dois acertos, essa estratégia deixa todos os desgastes passarem.

# Ajustar e prever classes

A regressão logística estima uma probabilidade de classe por meio de uma transformação logística. O método predict devolve um rótulo. A classe verdadeira de cada nova bateria será conhecida apenas pela inspeção.

### Demonstração

Ajuste um classificador nas primeiras oito baterias e preveja as quatro seguintes.

In [ ]:
from sklearn.linear_model import LogisticRegression
modelo = LogisticRegression().fit([[1],[2],[3],[4],[6],[7],[8],[9]], [0,0,0,0,1,1,1,1])
medidas = [[2],[4],[7],[9]]
print(modelo.predict(medidas).tolist())

O modelo prevê [0, 0, 1, 1]. A medição é a entrada disponível antes da inspeção.

### Variação

Veja a probabilidade para cada classe na bateria com medição 7.

In [ ]:
print(modelo.classes_.tolist(), modelo.predict_proba([[7]]).round(3).tolist())

Há duas probabilidades na ordem 0, 1. A soma antes de arredondar é 1.

### Aplicação

Simule um limiar de inspeção mais baixo, de 0,2. Compare quantas baterias seriam chamadas.

In [ ]:
prob = modelo.predict_proba(medidas)[:, 1]
print(int((prob >= 0.2).sum()), int((prob >= 0.5).sum()))

Baixar o limiar pode chamar mais baterias. Antes de adotá-lo, compare desgaste perdido e inspeções adicionais em validação.

# Ler uma matriz de confusão

Compare os rótulos da inspeção com os rótulos previstos. Na matriz do scikit-learn, linhas representam a classe verdadeira e colunas representam a prevista. Informe labels=[0, 1] para fixar a convenção inclusive quando uma classe estiver ausente no lote.

### Demonstração

Monte a matriz para as quatro baterias da validação.

In [ ]:
from sklearn.metrics import confusion_matrix
real, previsto = [0,1,0,1], [0,0,1,1]
print(confusion_matrix(real, previsto, labels=[0,1]).tolist())

A matriz é [[1, 1], [1, 1]]. Há um exemplo em cada combinação.

### Variação

Descubra a posição humana, contando a partir de 1, do falso negativo.

In [ ]:
print([i for i, (r, p) in enumerate(zip(real, previsto), 1) if r == 1 and p == 0])

A segunda bateria tinha desgaste e foi liberada.

### Aplicação

Altere apenas a segunda previsão para 1 e localize a célula que diminui e a que aumenta.

In [ ]:
print(confusion_matrix(real, [0,1,1,1], labels=[0,1]).tolist())

A matriz passa a [[1, 1], [0, 2]]. Um falso negativo passa a verdadeiro positivo.

# Comparar precisão e revocação

Precisão responde quantos dos alertas foram confirmados. Revocação responde quantos dos desgastes reais foram detectados. As duas medidas diferem pelo denominador. Declare que a classe positiva é desgaste e ligue o custo de cada erro à decisão.

### Demonstração

Calcule as métricas originais e depois as da política que inspeciona a segunda bateria.

In [ ]:
from sklearn.metrics import precision_score, recall_score
real = [0,1,0,1]
for p in ([0,0,1,1], [0,1,1,1]):
    print(round(precision_score(real, p), 3), recall_score(real, p))

O primeiro par é 0,5 e 0,5. O segundo é aproximadamente 0,667 e 1,0.

### Variação

Calcule a política que inspeciona todas.

In [ ]:
p = [1,1,1,1]
print(precision_score(real, p), recall_score(real, p))

Revocação atinge 1. Só metade dos quatro alertas tem desgaste confirmado.

### Aplicação

Se perder um desgaste custa 10 unidades e investigar um falso alerta custa 2, atribua custo à política original.

In [ ]:
custo = sum(10 * (r == 1 and p == 0) + 2 * (r == 0 and p == 1) for r, p in zip(real, [0,0,1,1]))
print(custo)

O custo é 12. A política que chama todas custa 4 neste lote. Quatro baterias não bastam para generalizar.

# Receber e auditar o lote

Retome o problema de prever duração antes da partida. O kit traz uma série sintética com uma viagem por dia, ordem conhecida e medidas ausentes. Antes de ajustar, confira os IDs, a unidade, o contrato das colunas e se os intervalos reservados serão suficientes para completar as três janelas.

### Demonstração

Leia a série sem acessar os alvos de teste. Verifique o formato de X e a unicidade.

In [ ]:
from laboratorio import carregar
ids, X, y = carregar("serie.csv")
print(len(set(ids)), X.shape, ids[99:102])

Há 120 IDs distintos e 2 atributos. S100 encerra o desenvolvimento e S101 começa a reserva.

### Variação

Conte distâncias ausentes só no desenvolvimento e confirme o tratamento do zero.

In [ ]:
import numpy as np
print(np.isnan(X[:100, 0]).sum(), (X[:100, 1] == 0).sum())

A primeira contagem indica quantas distâncias precisarão de imputação. A segunda conta viagens conhecidas sem paradas.

### Aplicação

Mostre que o intervalo reservado contém precisamente 20 IDs e nenhum deles ajudou a ajustar o modelo.

In [ ]:
print(len(ids[100:]), set(ids[:100]).isdisjoint(ids[100:]))

Imprime 20 e True. Registre essas fronteiras no relatório antes de comparar alternativas.

# Congelar o procedimento

Trabalhe agora no arquivo inicial/projeto.py do kit. Para cada alternativa, ajuste de novo nas primeiras 40, 60 e 80 linhas e meça o MAE no bloco seguinte. Use a menor média, reajuste uma vez nas primeiras 100 e registre o nome e as regras antes de avaliar os últimos dias.

### Demonstração

Compare sua implementação com a escolha reproduzida pela solução.

In [ ]:
from laboratorio import carregar
from solucao.projeto import selecionar
ids, X, y = carregar("serie.csv")
nome, modelo, medidas = selecionar(X[:100], y[:100])
print(nome, sorted(medidas))

As alternativas são mediana, ridge-1 e ridge-10. Com a série fornecida, vence ridge-1.

### Variação

Registre o contrato da previsão em uma estrutura que outra pessoa possa ler.

In [ ]:
ficha = {"modelo": nome, "fim_treino": ids[99], "atributos": ["km_planejados", "paradas_planejadas"], "metrica": "MAE (min)"}
print(ficha)

A ficha liga a escolha aos atributos permitidos e ao fim do desenvolvimento.

### Aplicação

Uma equipe prefere ridge-10 depois de ver o teste. Proponha um procedimento honesto para avaliar a mudança.

In [ ]:
print("Próximo dado de avaliação precisa ser posterior a", ids[-1])

A hipótese pode ser explorada, mas exige outro período futuro reservado para uma avaliação independente.

# Avaliar o período reservado

Com a escolha congelada, faça uma passagem pelos últimos vinte dias. Ajuste também a referência de mediana nas mesmas cem linhas. Calcule ambos os erros no mesmo intervalo. Evite voltar às escolhas com base nesse placar.

### Demonstração

Compare os resultados no período posterior e gere o gráfico de resíduos.

In [ ]:
from solucao.projeto import avaliar
p = avaliar()
print("Quantidade de previsões:", len(p))

A solução gera residuos.png e 20 previsões. O MAE final do selecionado é aproximadamente 3,41 min, e o da referência é 7,21 min.

### Variação

Determine a viagem com maior erro absoluto e guarde o sinal para interpretar o gráfico.

In [ ]:
import numpy as np
from laboratorio import carregar
ids, X, y = carregar("serie.csv")
residuos = y[100:] - p
i = np.abs(residuos).argmax()
print(ids[100:][i], round(residuos[i], 2))

O ID permite retomar a linha no CSV. Com a convenção real menos previsto, um valor positivo indica uma viagem mais demorada que o previsto.

### Aplicação

Faça a variação inédita do kit. Some 7 minutos a cada duração final sem alterar o modelo e teste a mudança dos resíduos.

In [ ]:
novos = (y[100:] + 7) - p
print(np.allclose(novos - residuos, 7))

Todos os resíduos sobem sete minutos. O valor absoluto e o MAE podem variar de maneira diferente em cada sinal.

# Comunicar resultados e limites

Entregue o script, a tabela com os três resultados de validação, o nome congelado, as duas métricas finais e o gráfico. Outra pessoa precisa conseguir repetir a avaliação em uma pasta limpa. Os dados têm ruído e uma mudança de duração no final. A experiência mede desempenho apenas nesse cenário sintético.

### Demonstração

Produza uma frase verificável com tamanho da reserva, unidade e métrica.

In [ ]:
from laboratorio import carregar
from solucao.projeto import selecionar
from sklearn.metrics import mean_absolute_error
ids, X, y = carregar("serie.csv")
nome, modelo, _ = selecionar(X[:100], y[:100])
p = modelo.predict(X[100:])
print(f"{nome}: MAE de {mean_absolute_error(y[100:], p):.2f} min em {len(p)} viagens reservadas.")

A frase identifica ridge-1, 3,41 minutos e 20 viagens. Acrescente que as viagens são sintéticas.

### Variação

Compare o MAE nas duas metades da reserva e diga por que os valores podem variar.

In [ ]:
for a, b in ((0,10), (10,20)):
    print(ids[100+a], round(mean_absolute_error(y[100+a:100+b], p[a:b]), 2))

Cada metade compara dez observações. Ruído, distâncias e paradas podem mudar a composição dos períodos.

### Aplicação

Proponha uma decisão acompanhada de uma condição para reavaliá-la e confira o ganho absoluto sobre a referência.

In [ ]:
from laboratorio import alternativas
ref = alternativas()["mediana"].fit(X[:100], y[:100]).predict(X[100:])
print(round(mean_absolute_error(y[100:], ref) - mean_absolute_error(y[100:], p), 2))

O ganho é cerca de 3,8 min de MAE neste período. Antes de usar o modelo em outra cidade, repita a avaliação em viagens representativas dela.